# Step 1 on Kaggle: convert iSign to keypoint shards (once, about 1 hour)

Same conversion as `colab/isign_convert_poses.ipynb`, but the shards are saved as this notebook's
**output** on Kaggle instead of Google Drive. The training notebook then uses that output as its input.

**Settings (right-hand panel) before running:**
- *Session options → Accelerator:* **None** (the conversion is download-bound; keep your GPU hours for training)
- *Session options → Internet:* **On** (needs a phone-verified Kaggle account)
- *Add-ons → Secrets:* add **`HF_TOKEN`** = your Hugging Face token (the account that accepted the iSign terms), and tick it for this notebook

Then click **Save Version → Save & Run All (Commit)**. It runs in the background; you can close the tab.

In [ ]:
!git clone -q https://github.com/sengarshivansh/ISL.git /tmp/ISL
!pip -q install pose-format stream-unzip

In [ ]:
import sys, shutil, os
sys.path.insert(0, '/tmp/ISL/colab')
import isign_pose_convert as C
from kaggle_secrets import UserSecretsClient
from huggingface_hub import hf_hub_download

TOKEN = UserSecretsClient().get_secret('HF_TOKEN')
OUT_DIR = '/kaggle/working/isign_pose_keypoints'   # everything in /kaggle/working becomes the output
os.makedirs(OUT_DIR, exist_ok=True)
shutil.copy(hf_hub_download(C.REPO, 'iSign_v1.1.csv', repo_type='dataset', token=TOKEN), OUT_DIR)
C.run(C.hf_part_chunks(TOKEN), OUT_DIR, target_fps=10, shard_size=500, tmp_dir='/tmp/isign_pose_work')

In [ ]:
import glob, json, numpy as np
shards = sorted(glob.glob(f'{OUT_DIR}/shard_*.npz'))
total = sum(len(np.load(s).files) for s in shards)
size = sum(os.path.getsize(s) for s in shards) / 1e9
print(f'{len(shards)} shards, {total} clips, {size:.1f} GB, '
      f'{json.load(open(f"{OUT_DIR}/layout.json"))["feat_dim"]} values per frame')
assert len(shards) == 255 and total > 127000, 'conversion incomplete'